# Lab 7.2 &mdash; Gather Context with Tools

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 4 &middot; Module 7 &mdash; Task Automation with AI Agents**

### What you'll do
- Wrap lookup_order and get_template as @tool functions
- Gather the order + the reply template BEFORE drafting
- See why gather-first prevents hallucinated specifics

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** the graded steps are **offline &amp; deterministic** (pure Python stdlib); the agent-assembly labs reuse the **LangChain-shaped shim** from Module 6. Advanced labs end with an **optional** cell that runs the **real** library. You are building the **email-drafting agent** &mdash; the client's Lab 4.1.

**Reference:** [Module 7 slides &mdash; Grounding the task in real data](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 7 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-07-02"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
A general model doesn't know **your** client's order or **your** reply templates &mdash; so the agent
must **gather context first, then draft** (deck slide 6). Gathering happens through **tools** over
your systems: an orders DB, a template store, the CRM. An agent that drafts before it gathers
**hallucinates specifics**; one that grounds every claim in retrieved context is accurate and
auditable.

In [ ]:
# --- LangChain-SHAPED shim: a tool has .name, .description (from the docstring), .args, .invoke() ---
import inspect
class Tool:
    def __init__(self, fn, name=None, description=None):
        self.fn = fn
        self.name = name or fn.__name__
        self.description = (description or inspect.getdoc(fn) or "").strip()
        self.args = list(inspect.signature(fn).parameters)
    def invoke(self, value):
        return self.fn(**value) if isinstance(value, dict) else self.fn(value)
    def __repr__(self):
        return "Tool(name=%r)" % self.name
def tool(fn):
    """The @tool decorator -- wrap a plain function into a Tool (mirrors langchain_core.tools.tool)."""
    return Tool(fn)

# The email agent's context sources: a small orders DB and a set of reply templates.
ORDERS = {
    "4471": {"id": "4471", "name": "Priya", "status": "shipped",    "eta": "Friday",    "carrier": "BlueDart"},
    "5090": {"id": "5090", "name": "Sam",   "status": "processing", "eta": "next week", "carrier": "-"},
}
TEMPLATES = {
    "delivery_delay": "Hi {name}, your order {id} has {status} and is due {eta}. Thanks for your patience.",
    "refund":         "Hi {name}, we've started the refund for order {id}. It will reflect in 5-7 days.",
}

print("orders on file :", list(ORDERS))
print("templates on file:", list(TEMPLATES))

## Your Turn
Complete the two gather tools and the `gather` step that pulls both before any drafting.

In [ ]:
@tool
def lookup_order(order_id):
    """Look up an order's status, ETA and carrier by id."""
    return ___   # TODO: the order dict for order_id, or {"status": "unknown"} if not found

@tool
def get_template(kind):
    """Fetch a reply template by kind, e.g. delivery_delay or refund."""
    return ___   # TODO: the template string for kind, or "" if none

def gather(order_id, kind):
    # gather FIRST: pull the order AND the template before we draft anything
    return {"order": ___, "template": ___}   # TODO: invoke each tool

try:
    print('order 4471 :', lookup_order.invoke('4471'))
    print('unknown    :', lookup_order.invoke('9999'))
    ctx = gather('4471', 'delivery_delay')
    print('gathered   :', ctx)
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("lookup_order finds a real order", lambda: lookup_order.invoke("4471")["status"] == "shipped")
expect_true("an unknown order degrades to 'unknown'", lambda: lookup_order.invoke("9999")["status"] == "unknown")
expect_true("get_template returns the delivery template", lambda: "{name}" in get_template.invoke("delivery_delay"))
expect_true("gather returns BOTH order and template", lambda: set(gather("4471", "delivery_delay")) == {"order", "template"})
expect_true("gather grounds on real data (ETA Friday)", lambda: gather("4471", "delivery_delay")["order"]["eta"] == "Friday")

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
Gather first, draft second. These are just Module-6 tools pointed at a real job -- the order and the template are the ground truth the draft will stand on.

[&#8592; All Module 7 labs](./index.html) &nbsp;&middot;&nbsp; [Module 7 slides](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>